# Get/Generate words for search

## 1 Importing needed packages

In [ ]:
import pandas as pd
import unicodedata
from tqdm.notebook import tqdm
from constants import DATA_PATH, TMP_PATH
from utils import check_and_create_file

tqdm.pandas()

## 2 Read/Preprocess data 


In [ ]:
names_df = pd.read_csv(
    "../data/tables/names.csv",
    sep=",",
    usecols=[
        "label:en",
        "gender",
        "label:el",
        "label:el:norm",
        "alternatives",
        "Nom Sg",
        "Gen Sg",
        "Dat Sg",
        "Akk Sg",
        "Voc Sg",
        "factgrid",
    ],
)


In [ ]:
names_df["type"] = "name"

Concat multiple dataframes into one big dataframe

In [ ]:
words_df = pd.concat(
    [names_df], ignore_index=True
)

words_df

### 2.1 Remove rows with empty greek label

In [ ]:
# names_df.dropna(subset=["label:el"], inplace=True, ignore_index=True)
# nominals_df.dropna(subset=["label:el"], inplace=True, ignore_index=True)

words_df.dropna(subset=["label:el"], inplace=True, ignore_index=True)

### 2.2 Remove diacritics and merge column data
As the transcripted text contains no accents and is completely lowercase, diacritcs have to be removed from data collected on persons. All cases and alternative spellings get merged into a column named 'variants' to remove identical forms


In [ ]:
def str_remove_diacritics(s: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFKD", s) if unicodedata.category(c) != "Mn"
    ).lower()


def df_remove_diacritics(df: pd.DataFrame, columns: list):
    # remove diacritics from all columns named in columns_to_process
    for col in columns:
        df[col] = df[col].progress_apply(
            lambda x: str_remove_diacritics(x) if pd.notnull(x) else x
        )


def columns_to_set(row) -> set:
    filtered_list = [e for elem in row if pd.notnull(elem) for e in elem.split(",")]
    return set(filtered_list)

In [ ]:
columns = [
    "label:el",
    "Nom Sg",
    "Gen Sg",
    "Dat Sg",
    "Akk Sg",
    "Voc Sg",
    "Nom Pl",
    "Gen Pl",
    "Dat Pl",
    "Akk Pl",
    "Voc Pl",
    "alternatives",
]

df_remove_diacritics(words_df, columns)
words_df["label:el:norm"] = words_df["label:el"]
# Applying the function to merge columns
words_df["variants"] = words_df[columns].progress_apply(columns_to_set, axis=1)

words_df

In [ ]:
# drop merged columns
words_df.drop(
    labels=[
        "label:el",
        "Nom Sg",
        "Gen Sg",
        "Dat Sg",
        "Akk Sg",
        "Voc Sg",
        "Nom Pl",
        "Gen Pl",
        "Dat Pl",
        "Akk Pl",
        "Voc Pl",
        "alternatives",
    ],
    axis=1,
    inplace=True,
)

### 2.3 Explode by name

In [ ]:
# add index for words
words_df["wordID"] = range(0, len(words_df))
words_df = words_df.rename(columns={"variants": "variant"})
# explode name_df by column variant
words_df = words_df.explode("variant").reset_index()
# Adding a new column 'variantID' with unique numbers for each variant (row)
words_df["variantID"] = range(0, len(words_df))

words_df

### 2.4 Remove factgrid link to keep entity numbers only

In [ ]:
# split factgrid string on ,
# Splitting strings in the column based on comma and converting them into sets
words_df["factgrid"] = (
    words_df["factgrid"].astype(str).progress_apply(lambda x: set(x.split(",")))
)
# Exploding the sets in the column
words_df = words_df.explode("factgrid")
# Removing the substring from all strings in the column
words_df["factgrid"] = words_df["factgrid"].str.replace(
    "https://database.factgrid.de/entity/", ""
)

### 2.5 Remove leading/trailing whitespaces

In [ ]:
words_df = words_df.progress_apply(
    lambda col: col.map(lambda x: x.strip() if isinstance(x, str) else x)
)

## 3 Write to file

In [ ]:
# drop not needed column index (as it has already been updated) from dataframe
words_df.drop(columns=["index"], inplace=True)

# set nan
words_df["factgrid"] = words_df["factgrid"].replace("nan", "NA")
words_df["gender"] = words_df["gender"].fillna("NA")
words_df["gender"] = words_df["gender"].replace(
    "?", "NA"
)  # TODO: this should be revised in the original table
words_df["label:en"] = words_df["label:en"].fillna("NA")
words_df["label:en"] = words_df["label:en"].str.replace(r"\?$", "", regex=True)

# write to csv file
check_and_create_file(TMP_PATH + "words.csv")
words_df.to_csv(TMP_PATH + "words.csv", index=False)